# 第33章　医療AIの倫理・バイアス・限界

**『医療診断支援AI開発　社会実装編 ― 臨床現場に届ける（社会実装編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-social

## 公平性を、実測する ― コードで群間差を出す

In [ ]:
import numpy as np
from statsmodels.stats.proportion import proportion_confint

def group_sensitivity(df, group_col):
    """群ごとに感度とWilson95%CIを出し、群間差（ギャップ）を返す"""
    rows = []
    pos = df[df.label == 1]                      # 陽性（病変あり）症例のみで感度を測る
    # 群の一覧は「元のdf」から取る。陽性を抽出した後で groupby すると、
    # 陽性が0件の群が結果から消え、その群を「評価した」つもりで見落とす。
    for g in sorted(df[group_col].dropna().unique()):
        sub = pos[pos[group_col] == g]
        n = len(sub)
        if n == 0:                                # 評価不能として明示的に残す
            rows.append({"群": g, "陽性例数": 0, "感度": None,
                         "95%CI": None, "状態": "評価不能（陽性症例なし）"})
            continue
        tp = int((sub.pred == 1).sum())
        sens = tp / n
        lo, hi = proportion_confint(tp, n, alpha=0.05, method="wilson")
        rows.append({"群": g, "陽性例数": n, "感度": sens,       # 丸めるのは表示のときだけ
                     "95%CI": f"[{lo:.2f}, {hi:.2f}]", "状態": "ok"})
    ok = [r for r in rows if r["感度"] is not None]
    out = sorted(rows, key=lambda r: (r["感度"] is None, r["感度"] or 0.0))
    # 群間差は、丸める前の値で計算する。評価できた群が1つ以下なら差は出さない。
    gap = (max(r["感度"] for r in ok) - min(r["感度"] for r in ok)) if len(ok) >= 2 else None
    return out, gap

# 使い方：装置メーカー別、年齢層別などgroup_colを替えて繰り返す
detail, gap = group_sensitivity(results, "device_maker")
print(detail, "\n感度の群間差 =", "評価不能（評価可能な群が2群未満）" if gap is None else f"{gap:.3f}")

## 患者への説明と同意 ― 開示の実務テンプレート

```text
【AIの利用について（患者向け説明の骨子）】
・何のために使うか：本検査では、画像の見逃しを減らす目的で、
  診断を支援するAIを補助的に用いることがあります。
・誰が判断するか：最終的な診断と治療方針は、担当医が責任をもって決定します。
  AIはあくまで支援の道具です。
・限界：AIは万能ではなく、誤ることがあります。特に（例：ごく小さな病変）では
  性能が下がることが分かっています。
・データの扱い：あなたの画像は、（施設の規程・匿名化・保管期間）に従って扱われます。
・選択の自由：AIの利用を望まない場合や、質問がある場合は、担当医にお伝えください。
・問い合わせ先：（部署・連絡先）
```